In [ ]:
# Output configuration
iout = 50
path = "mhd"

In [ ]:
import osyris
import numpy as np
import matplotlib.pyplot as plt

data = osyris.RamsesDataset(iout, path=path).load() 
boxlen = data.meta["boxlen"]

center = osyris.Vector(x=boxlen/2, y=boxlen/2, z=boxlen/2, unit=osyris.units("pc")) # Choose the center of the box


data["mesh"]["temperature"] = (data["mesh"]["pressure"] /  data["mesh"]["density"] /  (1* osyris.units("k_B")) * (1.4 * osyris.units("m_p"))).to("K")

osyris.map(
    data["mesh"].layer("density"),  # layer 1 : density
    norm="log", origin=center, direction="z", dx=boxlen*osyris.units("pc"))

osyris.map(
    data["mesh"].layer("temperature"),  # layer 1 : temperature
    norm="log", origin=center, direction="z", cmap="Reds_r", dx=boxlen*osyris.units("pc"))

osyris.map(
    data["mesh"].layer("B_field"),  # layer 1 : magnetic field
    norm="log", origin=center, direction="z", cmap="Purples_r", dx=boxlen*osyris.units("pc"))

### Plot distributions

You can also directly access the data and plot basic distribution information. Here is an example for a density PDF. You can adapt this to plot the velocity PDF and a phase plot.

In [ ]:
rho = data["mesh"]["density"].to("g/cm^3").values
rhomean = np.mean(rho) # works because uniform grid
n =  (data["mesh"]["density"] / (1.4 * osyris.units("m_p"))).to('1/cm^3').values

plt.figure()
plt.hist(np.log10(n), bins=100, histtype="step", lw=2)
plt.yscale("log")
plt.xlabel("log(n)")
plt.ylabel("DF")

In [ ]:
vel = data["mesh"]["velocity"].to("km/s").values
mass = data["mesh"]["mass"].to("M_sun").values
velmean = np.sum(vel.T*mass, axis=1)/  np.sum(mass)
veldisp = np.sum((vel -velmean).T**2 *mass, axis=1)/  np.sum(mass)


In [ ]:
sigma = np.sqrt(np.sum(veldisp**2, axis=0))/np.sqrt(3)
print(f"The velocity dispersion in {sigma} km/s")

In [ ]:
rho = data["mesh"]["density"].to("g/cm^3").values
n =  (data["mesh"]["density"] / (1.4 * osyris.units("m_p"))).to('1/cm^3').values
mass = data["mesh"]["density"].to("g/cm^3").values 
plt.figure()
plt.hist2d(np.log10(n), np.log10(data["mesh"]["pressure"].values), bins=100, weights=mass, cmap="inferno")
#plt.yscale("log")
#plt.xscale("log")
plt.xlabel("log(n)")
plt.ylabel("log(P)")

plt.figure()
plt.hist2d(np.log10(rho), np.log10(data["mesh"]["temperature"].values), bins=100, weights=mass, cmap="inferno")
#plt.yscale("log")
#plt.xscale("log")
plt.xlabel("log(rho)")
plt.ylabel("log(T)")

In [ ]:
max_cs = np.max(np.sqrt(data["mesh"]["pressure"]/data["mesh"]["density"])).to("km/s").values
print(f"The maximal sound speed is {max_cs} km/s")

## Build a movie

You will need to modify the file ram3 to point towards a valid location of ffmpeg, eg. `/usr/bin/ffmpeg`.
Then **rerun** the simulation. RAM parameters a set in a `config.ini` file. Read it, modify it if necessary and you can generate the movie with

In [ ]:
%%bash
cd mhd
python ../ram/ram3.py

## Power Spectra

For this part we will use yt and the file pspec.py

https://yt-project.org/

In [ ]:
%pip install yt tables astropy

In [ ]:
import yt

### Get a datacube from yt

In [ ]:
path = "mhd"
ds = yt.load("mhd/output_00050/info_00050.txt")

In [ ]:
levelmin = ds.parameters["levelmin"]
res = 1 << levelmin 
grid = ds.r[
    ::res*1j,
    ::res*1j,
    ::res*1j,
]

In [ ]:
fits = grid.to_fits_data(
    fields=[("gas", "density"), 
            ("gas", "pressure"), 
            ("gas", "velocity_x"), 
            ("gas", "velocity_y"),
            ("gas", "velocity_z")], length_unit="pc"
)
fits.writeto(f"{path}/{iout}", overwrite=True)

### Get power spectra

In [ ]:
import pspec
ps = pspec.pspec(path=path, iouts=[50], magnetic=True)

In [ ]:
kbins = ps[iout]["3d"]["velocity_c"]["kbins"]
kbins = 0.5*(kbins[:-1] + kbins[1:])
pspec_vs =  ps[iout]["3d"]["velocity_s"]["pspec"]
pspec_vc =  ps[iout]["3d"]["velocity_c"]["pspec"]


plt.figure()
plt.loglog(kbins,pspec_vs, label="solenoidal")
plt.loglog(kbins,pspec_vc, label="compressive")

plt.xlabel("k")
plt.ylabel("P")
plt.legend()